In [0]:
from pyspark.sql.functions import col, to_timestamp, when, to_date, upper, round

df_price_bronze = spark.table("crypto_sentiment_bronze_price")

df_price_silver = df_price_bronze.select(
    upper(col("symbol")).alias("symbol"),
    round(col("price"), 4).alias("price"),
    col("cmc_rank"),
    col("volume_24h").cast("int"),
    round(col("percent_change_1h"), 2).alias("percent_change_1h"),
    round(col("percent_change_24h"), 2).alias("percent_change_24h"),
    round(col("percent_change_7d"), 2).alias("percent_change_7d"),
    col("last_updated").cast("timestamp").alias("market_timestamp"),
    col("ingested_at").cast("timestamp").alias("ingested_at") 
)


df_price_silver.display()

symbol,price,cmc_rank,volume_24h,percent_change_1h,percent_change_24h,percent_change_7d,market_timestamp,ingested_at
LINK,9.2558,16,1061111421,-1.77,-2.92,-21.79,2026-02-04T15:06:00Z,2026-02-04T15:06:57Z
USDT,0.9983,3,2147483647,-0.02,-0.09,-0.04,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z
SOLANA,0.0,6108,11,-2.33,-8.76,-25.43,2026-02-04T15:06:00Z,2026-02-04T15:06:57Z
CMC20,152.9817,8853,4311513,-1.05,-3.78,-18.8,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z
AVAX,9.741,23,458010450,-1.32,-1.98,-19.4,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z
HYPE,33.0138,12,842737291,1.4,-4.99,-2.39,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z
DOGE,0.104,9,1968228130,-1.7,-1.85,-16.98,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z
SHIB,0.0,25,164180736,-0.86,-2.82,-15.0,2026-02-04T15:06:00Z,2026-02-04T15:06:57Z
USDC,0.9997,6,2147483647,-0.01,-0.01,-0.02,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z
BTC,74396.3382,1,2147483647,-0.65,-3.88,-17.05,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z


In [0]:
df_price_silver = df_price_silver.filter((col("price") > 0) & (col("volume_24h") > 0))

df_price_silver = df_price_silver.withColumn("market_date", to_date(col("market_timestamp")))

df_price_silver = df_price_silver.withColumn(
    "risk_category",
    when(col("cmc_rank") <= 10, "Blue Chip")
    .when((col("cmc_rank") > 10) & (col("cmc_rank") <= 50), "Mid Cap")
    .otherwise("Small Cap")
)

df_price_silver.display()

symbol,price,cmc_rank,volume_24h,percent_change_1h,percent_change_24h,percent_change_7d,market_timestamp,ingested_at,market_date,risk_category
LINK,9.2558,16,1061111421,-1.77,-2.92,-21.79,2026-02-04T15:06:00Z,2026-02-04T15:06:57Z,2026-02-04,Mid Cap
USDT,0.9983,3,2147483647,-0.02,-0.09,-0.04,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Blue Chip
CMC20,152.9817,8853,4311513,-1.05,-3.78,-18.8,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Small Cap
AVAX,9.741,23,458010450,-1.32,-1.98,-19.4,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Mid Cap
HYPE,33.0138,12,842737291,1.4,-4.99,-2.39,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Mid Cap
DOGE,0.104,9,1968228130,-1.7,-1.85,-16.98,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Blue Chip
USDC,0.9997,6,2147483647,-0.01,-0.01,-0.02,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Blue Chip
BTC,74396.3382,1,2147483647,-0.65,-3.88,-17.05,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Blue Chip
DOT,1.4524,32,211072388,-1.59,-3.06,-21.77,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Mid Cap
ETH,2169.8794,2,2147483647,-1.23,-4.34,-28.03,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Blue Chip


In [0]:
%sql
TRUNCATE TABLE crypto_sentiment_silver_price;

In [0]:
price_query = (
    df_price_silver
      .write
      .format("delta")
      .option("checkpointLocation", "/chk/crypto_sentiment_silver_price")
      .mode("append")
      .saveAsTable("crypto_sentiment_silver_price")
)

In [0]:

df_news_bronze = spark.table("crypto_sentiment_bronze_news")

df_news_silver = df_news_bronze.select(
    upper(col("symbol")).alias("symbol"),
    upper(col("name")).alias("name"),
    col("description").alias("news_headline"),
    col("prediction"),
    col("published_at").cast("timestamp").alias("news_timestamp")
)






In [0]:

df_news_silver = df_news_silver.withColumn("news_date", to_date(col("news_timestamp")))


df_news_silver = df_news_silver.withColumn(
                "sentiment_score",
                when(col("prediction") == "Positive", 1)
                .when(col("prediction") == "Negative", -1)
                .otherwise(0)
            )

In [0]:
df_news_silver.display()

symbol,name,news_headline,prediction,news_timestamp,news_date,sentiment_score
BTC,BITCOIN,"Bitcoinwell.com, a bitcoin-only platform, promotes Bitcoin as a tool for independence in discussions surrounding government policy.",Positive,2026-02-03T15:00:52Z,2026-02-03,1
BTC,BITCOIN,"Giacomo Zucco predicts a 'Bitcoin's 2026 Boom' driven by global upheavals, emphasizing scaling, privacy, and true adoption through solutions like ARK and Lightning Network.",Positive,2026-02-03T15:00:12Z,2026-02-03,1
BTC,BITCOIN,"Bitcoin's rebound near $79,000 eased fears surrounding MicroStrategy's cost basis, suggesting potential for further price recovery.",Positive,2026-02-03T15:00:00Z,2026-02-03,1
XLM,STELLAR,"Rails is leveraging Stellar-based smart contract vaults and on-chain proofs to make high-speed perpetuals more attractive to institutions, suggesting increased institutional adoption for Stellar.",Positive,2026-02-03T15:00:00Z,2026-02-03,1
USDC,USD COIN,"YC's Nemil Dalal announced that funding will be distributed in USDC across major blockchain networks, indicating increased utility and adoption for the stablecoin.",Positive,2026-02-03T15:00:00Z,2026-02-03,1
SHIB,SHIBA INU,"An analyst flagged a brutal downside scenario for Shiba Inu (SHIB), where a breakdown could lead to an 81% price drop.",Negative,2026-02-03T15:02:00Z,2026-02-03,-1
HYPE,HYPERLIQUID,Hyperliquid's expansion into prediction markets and options is anticipated to fuel the HYPE token's third leg of recovery.,Positive,2026-02-03T15:00:16Z,2026-02-03,1
DOGE,DOGECOIN,"Analyst Trader Tardigrade suggests Dogecoin (DOGE) may be preparing for another parabolic rally, citing historical Price Momentum Oscillator patterns.",Positive,2026-02-03T15:00:28Z,2026-02-03,1
ETH,ETHEREUM,"Ethereum is among the major blockchain networks chosen for USDC funding distribution, signaling continued relevance and use.",Positive,2026-02-03T15:00:00Z,2026-02-03,1
SOL,SOLANA,"Solana is among the major blockchain networks chosen for USDC funding distribution, signaling continued relevance and use.",Positive,2026-02-03T15:00:00Z,2026-02-03,1


In [0]:
%sql
TRUNCATE TABLE crypto_sentiment_silver_news;

In [0]:
news_query = (
    df_news_silver
      .write
      .format("delta")
      .option("checkpointLocation", "/chk/crypto_sentiment_silver_news")
      .mode("append")
      .saveAsTable("crypto_sentiment_silver_news")
)